# Current aviation roadmaps are not within planetary boundaries
Bastien Païs (1), Alexandre Gondran (2), Lorie Hamelin (3) and Florian Simatos (1)*\
(1) Fédération ENAC ISAE-SUPAERO ONERA, Université de Toulouse, France\
(2) ENAC, French Civil Aviation University, Université de Toulouse, Toulouse, France\
(3) TBI, Université de Toulouse, CNRS, INRAE, INSA, Toulouse, France\
*Corresponding author: florian.simatos@isae.fr

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

from pais2026.main import run_pipeline, export_core_results

## Run with climate-change selected impacts

In [ ]:
results_cc = run_pipeline(
    data_dir='../data',
    impacts_file='1_Impact_per_MJ_CC_selection.xlsx',
    method='FHN',
    include_resources=True,
    traffic_growth=0.037,
    efficiency_gain=0.015,
)
dict_AESA = results_cc["dict_aesa"]
from pais2026.downscaling import PB_aviation_FHN, PB_aviation_GDP


## Run with PB-selected impacts

In [ ]:
results_pb = run_pipeline(
    data_dir='../data',
    impacts_file='2_Impact_per_MJ_PB_selection.xlsx',
    method='FHN',
    include_resources=True,
    traffic_growth=0.037,
    efficiency_gain=0.015
)
dict_AESA_PB = results_pb["dict_aesa"]


# Influence of traffic growth

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

traffic_growth_values = np.linspace(-0.10, 0.10, 100)

scenarios = ['Baseline', 'CR1', 'CR2', 'CR3', 'CR4', 'MTS1', 'MTS2', 'MTS3', 'MTS4']
PBs = ['RF', 'BItot', 'N', 'P', 'FWU', 'SOD']

results_growth = {
    scenario: {PB: 0.10 for PB in PBs}
    for scenario in scenarios
}

def total_aesa_2050(dict_aesa, scenario, PB):
    fuel_types = ['fk_historical', 'fk', 'efuels', 'atj', 'ft', 'hefa', 'lh2']
    return sum(
        dict_aesa[scenario][fuel_type][PB][-1]
        for fuel_type in fuel_types
        if fuel_type in dict_aesa[scenario] and PB in dict_aesa[scenario][fuel_type]
    )

for traffic_growth in traffic_growth_values:
    out = run_pipeline(
        data_dir='../data',
        impacts_file='1_Impact_per_MJ_CC_selection.xlsx',
        method='GDP',  # pour reproduire ton ancien calcul GDP
        include_resources=False,
        traffic_growth=traffic_growth,
        efficiency_gain=0.015,
    )

    dict_aesa_tmp = out["dict_aesa"]

    for scenario in scenarios:
        for PB in PBs:
            value = total_aesa_2050(dict_aesa_tmp, scenario, PB)

            if value > 1 and traffic_growth < results_growth[scenario][PB]:
                results_growth[scenario][PB] = traffic_growth

## Export

In [ ]:
export_core_results(results_cc, output_dir='../outputs', prefix='CC_selection')
export_core_results(results_pb, output_dir='../outputs', prefix='PB_selection')

# Plots

## Scenarios

### Figure 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# Récupération des scénarios depuis les sorties du pipeline
scenario_mix = results_cc["scenarios"]

years = np.arange(2023, 2051)

scenarios_to_plot = [
    "CR1", "CR2", "CR3", "CR4",
    "MTS1", "MTS2", "MTS3"
]

fuel_types = ["fk", "efuels", "atj", "ft", "hefa", "lh2"]

fuel_labels = {
    "fk": "Fossil kerosene",
    "efuels": "E-fuels",
    "atj": "ATJ",
    "ft": "FT",
    "hefa": "HEFA",
    "lh2": "LH2",
}

colors = {
    "fk": "#4d4d4dff",
    "efuels": "#0072b2ff",
    "atj": "#cc79a7ff",
    "ft": "#56b4e9ff",
    "hefa": "#e69f00ff",
    "lh2": "#8c2d04ff",
}

title_subplots={
    "CR1": "Roadmap-70",
    "CR2": "Roadmap-30-Bio",
    "CR3": "Roadmap-0",
    "CR4": "Roadmap-30-Efuel",
    "MTS1": "Bio-Adv",
    "MTS2": "Bio-HEFA",
    "MTS3": "Efuel-grid and Efuel-RES",
}

fig, axes = plt.subplots(2, 4, figsize=(14, 7), sharex=True, sharey=True)
axes = axes.flatten()

for ax, scenario in zip(axes, scenarios_to_plot):

    data = np.array([
        scenario_mix[scenario][fuel]
        for fuel in fuel_types
    ])

    total = data.sum(axis=0)
    shares = np.divide(data, total, out=np.zeros_like(data), where=total != 0)

    ax.stackplot(
        years,
        shares,
        labels=[fuel_labels[f] for f in fuel_types],
        colors=[colors[f] for f in fuel_types],
        edgecolor="black",
        linewidth=0.5
    )

    ax.set_title(title_subplots.get(scenario, scenario))
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    #ax.grid(axis="y", alpha=0.3)
    ax.set_xlim(2023, 2050)

# for ax in axes[-2:]:
#     ax.set_xlabel("Year")

# for ax in axes[::2]:
#     ax.set_ylabel("Fuel mix share")

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="lower left",
    bbox_to_anchor=(0.78, 0.2),
    ncol=2,
    frameon=False,
)

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

## AESA

### Figure 3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# --- PARAMETERS ---
scenarios = ['Baseline', 'CR1', 'CR2', 'CR3', 'CR4', 'MTS1', 'MTS2', 'MTS3', 'MTS4']
fuel_types = ['fk_historical', 'fk', 'efuels', 'atj', 'ft', 'hefa', 'lh2']
PBs = ['RF', 'BItot', 'N', 'P', 'FWU', 'SOD']

impact_name = [
    'Climate change', 'Biosphere integrity',
    'Nitrogen cycle', 'Phosphorus cycle',
    'Freshwater use', 'Stratospheric ozone depletion'
]

colors = [
            # Net
    '#9e9e9eff',      # Fossil historical
    '#4d4d4dff',      # Fossil
    '#0072b2ff',      # E-fuels
    '#cc79a7ff',      # ATJ
    '#56b4e9ff',      # FT
    '#e69f00ff',      # HEFA
    '#8c2d04ff'       # LH2
]

xlabels = [
    'Baseline', 'Roadmap-70', 'Roadmap-30-Bio',
    'Roadmap-0', 'Roadmap-30-Efuel',
    'Bio-Adv', 'Bio-HEFA', 'Efuel-grid', 'Efuel-RES'
]

# --- FIGURE ---
fig, axes = plt.subplots(3, 2, figsize=(17.1, 19.2), sharex=True)
axes = axes.flatten()

indices = ['a', 'b', 'c', 'd', 'e', 'f']

for i, ax in enumerate(axes):
    ax.text(0, 1.02, f'{indices[i]})', transform=ax.transAxes, size=12, weight='bold')

# --- LOOP OVER PBs ---
for j, PB in enumerate(PBs):

    ax = axes[j]

    # Background zones
    ax.axvspan(-0.75, 0.5, color='lightgrey', alpha=0.5)
    ax.axvspan(0.5, 4.5, color='lightyellow', alpha=1)
    ax.axvspan(4.5, 8.75, color='lightsalmon', alpha=0.5)

    ax.set_xlim(-0.75, 8.75)

    bottom_pos = np.zeros(len(scenarios))
    bottom_neg = np.zeros(len(scenarios))

    # --- STACKED BARS ---
    for i, fuel_type in enumerate(fuel_types):

        values = [dict_AESA[s][fuel_type][PB][27] for s in scenarios]

        if PB == 'RF':
            pos = np.maximum(values, 0)
            neg = np.minimum(values, 0)

            ax.bar(scenarios, pos, bottom=bottom_pos, color=colors[i], edgecolor='k', linewidth=0.5)
            ax.bar(scenarios, neg, bottom=bottom_neg, color=colors[i], edgecolor='k', linewidth=0.5)

            bottom_pos += pos
            bottom_neg += neg

            ax.axhline(0, color='black')

        else:
            ax.bar(scenarios, values, bottom=bottom_pos, color=colors[i], edgecolor='k', linewidth=0.5)
            bottom_pos += values

    # Net sum for RF
    if PB == 'RF':
        net = bottom_pos + bottom_neg
        ax.plot(
            scenarios, net,
            marker='d', markerfacecolor='w',
            linestyle='None', markeredgecolor='k'
        )

    # --- AXES ---
    ax.set_xticklabels(xlabels, rotation=90)
    ax.set_ylabel('Normalized impact under \nethic-based allocation')
    ax.set_title(impact_name[j])

    # Scale
    ax.set_ylim(
        bottom=ax.get_ylim()[0] * 1.2,
        top=ax.get_ylim()[1] * 1.2
    )

    # --- SECOND AXIS ---
    ax2 = ax.twinx()

    ratio = PB_aviation_FHN[PB] / PB_aviation_GDP[PB]

    ax2.set_ylim(
        ax.get_ylim()[0] * ratio,
        ax.get_ylim()[1] * ratio
    )

    # Format %
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax2.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{y:.0%}'))

    ax2.set_ylabel('Normalized impact under \neconomic-based allocation')

    # Threshold lines
    ax.axhline(1, color='lawngreen', linestyle='--', linewidth=1.5)
    ax2.axhline(1, color='red', linestyle='--', linewidth=1.5)

    # Text zones
    ax.text(-0.125, ax.get_ylim()[1]*0.95, 'Baseline\n scenario', ha='center')
    ax.text(2.5,   ax.get_ylim()[1]*0.95, 'Current roadmaps', ha='center')
    ax.text(6.625, ax.get_ylim()[1]*0.95, 'Mono-type scenarios', ha='center')

# --- LEGEND ---
legend_elements = [
    plt.Line2D([0],[0], color='w', marker='d', linestyle='None', markeredgecolor='k', label='Net'),
    plt.Line2D([0],[0], color='#9e9e9eff', lw=6, label='Fossil (historical)'),
    plt.Line2D([0],[0], color='#4d4d4dff', lw=6, label='Fossil'),
    plt.Line2D([0],[0], color='#0072b2ff', lw=6, label='E-fuels'),
    plt.Line2D([0],[0], color='#cc79a7ff', lw=6, label='AtJ'),
    plt.Line2D([0],[0], color='#56b4e9ff', lw=6, label='FT'),
    plt.Line2D([0],[0], color='#e69f00ff', lw=6, label='HEFA'),
    plt.Line2D([0],[0], color='#8c2d04ff', lw=6, label='LH2'),
    plt.Line2D([0],[0], color='lawngreen', lw=2, linestyle='--', label='Share of SOS under ethic-based allocation'),
    plt.Line2D([0],[0], color='red', lw=2, linestyle='--', label='Share of SOS under economic-based allocation'),
]

fig.legend(
    handles=legend_elements,
    loc='center',
    ncol=5,
    bbox_to_anchor=(0.5, -0.04),
    title='Legend'
)

plt.tight_layout()
plt.show()

## Resources

### Figure 4

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Outputs from modules
dict_biomass = results_cc["dict_biomass"]
dict_electricity = results_cc["dict_electricity"]
land_occupation_biomass = results_cc["land_occupation_biomass"]
land_occupation_electricity = results_cc["land_occupation_electricity"]
freshwater_withdrawal = results_cc["freshwater_withdrawal"]
DAC_results = results_cc["dac_results"]

scenarios = ['Baseline', 'CR1', 'CR2', 'CR3', 'CR4', 'MTS1', 'MTS2', 'MTS3', 'MTS4']
xlabels = [
    'Baseline', 'Roadmap-70', 'Roadmap-30-Bio',
    'Roadmap-0', 'Roadmap-30-Efuel',
    'Bio-Adv', 'Bio-HEFA', 'Efuel-grid', 'Efuel-RES'
]
fuel_types = ['FK', 'efuels', 'ATJ', 'FT', 'HEFA', 'LH2']

colors = [
     '#4d4d4dff',      # Fossil
    '#0072b2ff',      # E-fuels
    '#cc79a7ff',      # ATJ
    '#56b4e9ff',      # FT
    '#e69f00ff',      # HEFA
    '#8c2d04ff'       # LH2
]

fig, axes = plt.subplots(2, 2, figsize=(20, 15), sharex=True)
indices = np.arange(len(scenarios))
barwidth = 0.40

for ax in axes.flatten():
    ax.axvspan(-0.75, 0.5, color='lightgrey', alpha=0.5)
    ax.axvspan(0.5, 4.5, color='lightyellow', alpha=1)
    ax.axvspan(4.5, 8.75, color='lightsalmon', alpha=0.5)
    ax.set_xlim(-0.75, 8.75)

axes[0, 0].text(0, 1.01, 'a)', transform=axes[0, 0].transAxes, size=15)
axes[0, 1].text(0, 1.01, 'b)', transform=axes[0, 1].transAxes, size=15)
axes[1, 0].text(0, 1.01, 'c)', transform=axes[1, 0].transAxes, size=15)
axes[1, 1].text(0, 1.01, 'd)', transform=axes[1, 1].transAxes, size=15)

# a) Electricity and biomass
ax = axes[0, 0]
bottom_biomass = np.zeros(len(scenarios))
bottom_electricity = np.zeros(len(scenarios))

for i, fuel_type in enumerate(fuel_types):

    values_biomass = []
    values_elec = []

    for s in scenarios:
        bio = dict_biomass[s].get(fuel_type, 0)
        elec = dict_electricity[s].get(fuel_type, 0)

        if isinstance(bio, dict):
            bio = bio.get("value", 0)

        if isinstance(elec, dict):
            elec = elec.get("value", 0)

        values_biomass.append(bio)
        values_elec.append(elec * 3.6)

    ax.bar(
        indices + barwidth / 2,
        values_biomass,
        bottom=bottom_biomass,
        color=colors[i],
        width=barwidth,
        edgecolor='black'
    )

    ax.bar(
        indices - barwidth / 2,
        values_elec,
        bottom=bottom_electricity,
        color=colors[i],
        width=barwidth,
        hatch='//',
        edgecolor='black'
    )

    bottom_biomass += np.array(values_biomass)
    bottom_electricity += np.array(values_elec)

ax.set_ylabel('Final electricity and biomass demand (EJ)')
ax.set_title('2050 aviation biomass and electricity demand')
ax.set_xticks(indices)
ax.set_xticklabels(xlabels, rotation=90)
ax.set_ylim(0, ax.get_ylim()[1] * 1.05)
ax.legend(
    handles=[
        Patch(facecolor='white', edgecolor='black', label='Biomass'),
        Patch(facecolor='white', edgecolor='black', hatch='//', label='Electricity')
    ],
    loc='upper left'
)

# b) Land occupation
ax = axes[0, 1]
bottom = np.zeros(len(scenarios))

for i, fuel_type in enumerate(fuel_types):
    values = [land_occupation_biomass[s][fuel_type] * 1e-6 for s in scenarios]
    ax.bar(indices, values, bottom=bottom, color=colors[i], edgecolor='black')
    bottom += np.array(values)

for i, fuel_type in enumerate(fuel_types):
    values = [land_occupation_electricity[s][fuel_type] * 1e-6 for s in scenarios]
    ax.bar(indices, values, bottom=bottom, color=colors[i], hatch='//', edgecolor='black')
    bottom += np.array(values)

ax.set_ylabel('Land occupation (mega-km²)')
ax.set_title('Land occupation by 2050 aviation')
ax.set_xticks(indices)
ax.set_xticklabels(xlabels, rotation=90)
ax.set_ylim(0, ax.get_ylim()[1] * 1.05)
ax.legend(
    handles=[
        Patch(facecolor='white', edgecolor='black', label='Biomass cultivation'),
        Patch(facecolor='white', edgecolor='black', hatch='//', label='Electricity production')
    ],
    loc='upper left'
)

# c) Freshwater withdrawal
ax = axes[1, 0]
bottom = np.zeros(len(scenarios))

for i, fuel_type in enumerate(fuel_types):
    values = [freshwater_withdrawal[s][fuel_type] for s in scenarios]
    ax.bar(indices, values, bottom=bottom, color=colors[i], edgecolor='black')
    bottom += np.array(values)

ax.set_ylabel('Freshwater withdrawal (km³)')
ax.set_title('Freshwater withdrawal by 2050 aviation')
ax.set_xticks(indices)
ax.set_xticklabels(xlabels, rotation=90)
ax.set_ylim(0, ax.get_ylim()[1] * 1.05)

# d) DAC
ax = axes[1, 1]
values = [DAC_results[s] for s in scenarios]

ax.bar(indices, values, color='#0072b2ff', edgecolor='black')
ax.set_ylabel('Direct air capture (MtCO₂)')
ax.set_title('Direct air capture by 2050 aviation')
ax.set_xticks(indices)
ax.set_xticklabels(xlabels, rotation=90)
ax.set_ylim(0, ax.get_ylim()[1] * 1.05)

# Scenario group labels
for ax in axes.flatten():
    y = ax.get_ylim()[1]
    ax.text(-0.125, y * 0.95, 'Baseline\nscenario', ha='center', va='center', fontsize=10)
    ax.text(2.5, y * 0.98, 'Current roadmaps (CR)', ha='center', va='center', fontsize=10)
    ax.text(6.625, y * 0.98, 'Mono-type scenarios (MTS)', ha='center', va='center', fontsize=10)

# Common legend
legend_handles = [
    Patch(facecolor=colors[0], edgecolor='black', label='Fossil kerosene'),
    Patch(facecolor=colors[1], edgecolor='black', label='E-fuels'),
    Patch(facecolor=colors[2], edgecolor='black', label='ATJ'),
    Patch(facecolor=colors[3], edgecolor='black', label='FT'),
    Patch(facecolor=colors[4], edgecolor='black', label='HEFA'),
    Patch(facecolor=colors[5], edgecolor='black', label='LH2'),
]

fig.legend(
    handles=legend_handles,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.01),
    ncol=6,
    fontsize='large'
)

plt.tight_layout()
plt.show()

## Traffic

### Figure 5

In [ ]:
markers = {
    'RF': '$RF$',
    'BItot': '$BI$',
    'N': '$N$',
    'P': '$P$',
    'FWU': '$FWU$',
    'SOD': '$SOD$',
}

colors = {
    'RF': '#fee08b',
    'BItot': '#fdae61',
    'N': '#f46d43',
    'P': '#d73027',
    'FWU': '#a50026',
    'SOD': '#7f0000',
}

xlabels = [
    'Baseline', 'Roadmap-70', 'Roadmap-30-Bio',
    'Roadmap-0', 'Roadmap-30-Efuel',
    'Bio-Adv', 'Bio-HEFA', 'Efuel-grid', 'Efuel-RES'
]

fig, ax = plt.subplots(figsize=(12, 8))

for scenario in scenarios:
    bottom = -0.10

    for PB in PBs:
        value = results_growth[scenario][PB]

        ax.bar(
            scenario,
            value - bottom,
            bottom=bottom,
            edgecolor='black',
            color=colors[PB]
        )

        ax.scatter(
            scenario,
            value,
            color='black',
            marker=markers[PB]
        )

        bottom = value

ax.axhspan(-0.10, -0.125, color='lightgrey', alpha=0.5)
ax.axhspan(-0.125, -0.15, color='lightgrey', alpha=0.5)

ax.axhline(y=-0.10, color='black')
ax.axhline(y=-0.125, color='black')
ax.axhline(y=-0.15, color='black')

ax.axhline(y=0.037, color='red', linestyle='--', label='Reference traffic growth')

ax.text(-0.9, -0.1375, 'Historical', ha='right', va='center')
ax.text(-0.9, -0.1125, 'Below -10.0%', ha='right', va='center')

ax.set_xticks(range(len(scenarios)))
ax.set_xticklabels(xlabels, rotation=90)

ax.set_yticks(np.arange(-0.10, 0.125, 0.025))
ax.set_ylim(-0.15, 0.10)
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{y:.1%}'))

ax.set_ylabel('Maximum traffic growth rate compatible with AESA threshold')
ax.set_title('Sensitivity to traffic growth rate')

plt.tight_layout()
plt.show()